In [1]:
!pip install selenium


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install chromedriver-autoinstaller

In [2]:
import csv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os

In [3]:
class WebDriverContext:
    def __init__(self, driver_path, options=None):
        self.driver_path = driver_path
        self.options = options

    def __enter__(self):
        service = Service(self.driver_path)
        self.driver = webdriver.Chrome(service=service, options=self.options)
        return self.driver

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.driver.quit()

In [4]:
chromedriver_path = r'C:\Users\KunalMajumdar\OneDrive - EPAM\GIT Epam\Personal\dqe-automation\Selenium Introduction\chromedriver-win64_v147\chromedriver-win64\chromedriver.exe'
html_report_path = r'C:\Users\KunalMajumdar\OneDrive - EPAM\GIT Epam\Personal\dqe-automation\generated_report\report.html'
os.chdir(r'C:\Users\KunalMajumdar\OneDrive - EPAM\GIT Epam\Personal\dqe-automation\Selenium Introduction')

In [5]:
with WebDriverContext(chromedriver_path) as driver:
    driver.get(html_report_path)

    try:
        table = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.CLASS_NAME, "table"))
        )

        columns = table.find_elements(By.CLASS_NAME, "y-column")
        headers = []
        data_columns = []

        for col in columns:
            header = col.find_element(By.ID, "header").text.strip()
            headers.append(header)
            cells = [cell.text for cell in col.find_elements(By.CLASS_NAME, "cell-text") if cell.text != header]
            data_columns.append(cells)

        rows = list(zip(*data_columns))

        with open('table.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)
        print("Table data saved to table.csv")

    except Exception as e:
        print(f"Error extracting table: {e}")

Table data saved to table.csv


In [6]:
def extract_doughnut_data(driver, doughnut, idx):
    labels = doughnut.find_elements(By.CSS_SELECTOR, "text.slicetext[data-notex='1']")
    chart_data = []
    for label in labels:
        tspans = label.find_elements(By.TAG_NAME, "tspan")
        if len(tspans) >= 2:
            category = tspans[0].text
            value = tspans[1].text
            chart_data.append([category, value])
    with open(f'doughnut{idx}.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Category', 'Value'])
        writer.writerows(chart_data)

with WebDriverContext(chromedriver_path) as driver:
    driver.get(html_report_path)

    try:
        doughnut = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "pielayer"))
        )

        driver.save_screenshot('screenshot0.png')
        extract_doughnut_data(driver, doughnut, 0)

        legend_box = driver.find_element(By.CLASS_NAME, "scrollbox")
        legend_items = legend_box.find_elements(By.CLASS_NAME, "traces")

        for idx, item in enumerate(legend_items, start=1):
            try:
                # Get current labels' text before clicking
                old_labels = doughnut.find_elements(By.CSS_SELECTOR, "text.slicetext[data-notex='1']")
                old_text = [label.text for label in old_labels]

                item.click()

                # Wait for the chart to update by checking that the text changes
                WebDriverWait(driver, 5).until(
                    lambda d: [
                        label.text for label in d.find_element(By.CLASS_NAME, "pielayer")
                        .find_elements(By.CSS_SELECTOR, "text.slicetext[data-notex='1']")
                    ] != old_text
                )

                doughnut = driver.find_element(By.CLASS_NAME, "pielayer")
                driver.save_screenshot(f'screenshot{idx}.png')
                extract_doughnut_data(driver, doughnut, idx)
            except Exception as e:
                print(f"Error with doughnut filter {idx}: {e}")

        print("Doughnut chart screenshots and CSVs saved.")

    except Exception as e:
        print(f"Error extracting doughnut chart: {e}")

Doughnut chart screenshots and CSVs saved.


In [7]:
def extract_doughnut_data(driver, doughnut, idx, filter_name):
    labels = doughnut.find_elements(By.CSS_SELECTOR, "text.slicetext[data-notex='1']")
    chart_data = []
    for label in labels:
        tspans = label.find_elements(By.TAG_NAME, "tspan")
        if len(tspans) >= 2:
            category = tspans[0].text
            value = tspans[1].text
            chart_data.append([category, value])
    filename = f'doughnut{idx}_{filter_name}.csv'
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Category', 'Value'])
        writer.writerows(chart_data)

with WebDriverContext(chromedriver_path) as driver:
    driver.get(html_report_path)

    try:
        doughnut = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "pielayer"))
        )

        # Initial screenshot and CSV (unfiltered)
        driver.save_screenshot('screenshot0_unfiltered.png')
        extract_doughnut_data(driver, doughnut, 0, 'unfiltered')

        legend_box = driver.find_element(By.CLASS_NAME, "scrollbox")
        legend_items = legend_box.find_elements(By.CLASS_NAME, "traces")

        for idx, item in enumerate(legend_items, start=1):
            try:
                # Get filter name, clean for filename
                filter_name = item.text.strip().replace(" ", "_")
                if not filter_name:
                    filter_name = f"filter{idx}"

                # Get current labels' text before clicking
                old_labels = doughnut.find_elements(By.CSS_SELECTOR, "text.slicetext[data-notex='1']")
                old_text = [label.text for label in old_labels]

                item.click()

                # Wait for the chart to update by checking that the text changes
                WebDriverWait(driver, 5).until(
                    lambda d: [
                        label.text for label in d.find_element(By.CLASS_NAME, "pielayer")
                        .find_elements(By.CSS_SELECTOR, "text.slicetext[data-notex='1']")
                    ] != old_text
                )

                doughnut = driver.find_element(By.CLASS_NAME, "pielayer")
                screenshot_filename = f'screenshot{idx}_{filter_name}.png'
                driver.save_screenshot(screenshot_filename)
                extract_doughnut_data(driver, doughnut, idx, filter_name)
            except Exception as e:
                print(f"Error with doughnut filter {idx}: {e}")

        print("Doughnut chart screenshots and CSVs saved.")

    except Exception as e:
        print(f"Error extracting doughnut chart: {e}")

Doughnut chart screenshots and CSVs saved.
